# Bronze Layer: Extraction

**Target Tables:**
- **Read:** `bronze_general_search` (SQLite `scraping_database.db`)
- **Write:** `bronze_ad_links` / Raw HTML storage (SQLite / Parquet)

**Objective:**
This notebook takes the list of URLs discovered in the `discover.ipynb` notebook and fetches the detailed HTML/JSON data for each ad. It filters the target regions based on the metadata to avoid unnecessary requests and extracts the raw content of each advertisement, saving it to the Bronze layer without heavy transformations.

**To-Do:**
- Implement the detailed extraction logic for OLX.
- Implement rate-limiting, retries, and anti-blocking strategies.
- Decide on the storage format for raw HTML/JSON (e.g., Parquet or Blob storage).

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[2])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import insert, select, update
from sqlalchemy.orm import Session
import polars as pl

# Importing from 'app' module
from app.services import OLXCrawler
from app.config import db_engine
from app.models import GeneralSearch, InformationExtraction
from app.utils import read_search_locations_metadata

In [2]:
# List to store the pages' data
extraction_data_list = []

# List to store the selled/deleted products
not_available_products = []

In [3]:
# Region data for filtered results due to limits
brazil_data_location = (
    read_search_locations_metadata()
    .get('search_locations', {})
    .get('brazil', {})
)

stores = (
    brazil_data_location
    .get('stores', [])
)

regions = (
    brazil_data_location
    .get('regions', [])
)

full_name_list = [region.get('full_name') for region in regions]
abreviation_list = [region.get('abreviation') for region in regions]

In [4]:
# Reading values
with db_engine.connect() as connection:
    # Removing undesired regions
    df_olx = (
        pl.read_database(
            select(GeneralSearch)
            .where(
                (GeneralSearch.region.in_(abreviation_list))
                & (GeneralSearch.store.is_('OLX'))
            ),
            connection=connection
        )
    )


In [5]:
df_olx = df_olx.head(100)

In [6]:
olx_crawler = OLXCrawler(base_data_list=extraction_data_list)
await olx_crawler.scrap_specific_information(
    df=df_olx,
    not_available_products=not_available_products
)

INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-go-14-full-hd-windows-11-excelente-estado-1520113274 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-go-128gb-excelente-estado-windows-11-1519419168 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://rj.olx.com.br/rio-de-janeiro-e-regiao/informatica/notebooks/vendo-notebook-samsung-galaxy-booker-go-1519193107 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://rs.olx.com.br/regioes-de-porto-alegre-torres-e-santa-cruz-do-sul/informatica/notebooks/notebook-samsung-galaxy-book-go-1509489854 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-samsung-galaxy-book-go-1519145866 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/samsung-galaxy-book-go-15

Unexpected error: Client error '403 Forbidden' for url 'https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3-15iml05-1518544661'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403


INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3-1519777324 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/vale-do-paraiba-e-litoral-norte/informatica/notebooks/notebook-lenovo-ideapad-3-ryzen-7-5700u-12gb-ram-ssd-512gb-excelente-estado-1518257500 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3-intel-core-i5-11-geracao-16gb-1518733276 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3-15iml05-1518544661 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://sp.olx.com.br/sao-paulo-e-regiao/informatica/notebooks/notebook-lenovo-ideapad-3-15igl05-1517762423 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://rs.olx.com.br/regioes-de-porto-alegre-torres-e-santa-cruz-do-sul/informatica/notebooks/lenovo-ideapad-3-15igl05-1517703

In [7]:
 # Creating dataframe 
extraction_data_df = pl.DataFrame(extraction_data_list)
updated_df = pl.DataFrame(not_available_products)

# Saving on sqlite 
if not extraction_data_df.is_empty():
    with Session(db_engine) as session:
        # Insert
        session.execute(
            insert(InformationExtraction), 
            extraction_data_df.to_dicts()
        )
        
        # Update
        if not updated_df.is_empty():
            session.execute(
                update(GeneralSearch), 
                updated_df.to_dicts()
            )
        else:
            print("No data found to update.")
        session.commit()
else:
    print("No data found to insert.")

No data found to update.
